# exp-back composite — cx4: exp_back — use cached out instead of recomputing exp(x)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `exp-back`, `backward-fn-signature`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "exp-back"
DD_ATOM_IDS = ["exp-back", "backward-fn-signature"]
DD_SUBTOPICS = ["Backprop: exp_back", "Backprop: backward fn signature"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `exp_back` via cached `out` — two atoms in one one-liner

1. **`backward-fn-signature`** — every back fn takes `(grad_out, out, *args)`.
   The `out` slot is the cached forward result; it's there *specifically* so
   that back fns can reuse it instead of recomputing the activation.
2. **`exp-back`** — the local derivative of `exp(x)` is `exp(x)`, which IS
   `out`. So the chain rule collapses to `grad_in = grad_out * out`. No
   second exp call needed.

Composition: the signature *gives* you `out`, and the math *requires* you to
use it. If you instead wrote `grad_out * t.exp(x)` you'd get the same number
(modulo float rounding) but pay for the activation twice. The test below
passes a deliberately WRONG `out` to catch implementations that secretly
recompute `exp(x)`.


### Composite Exercise — exp_back — use cached out instead of recomputing exp(x)

**Atoms exercised together**: `exp-back`, `backward-fn-signature`

Implement `cx4_exp_back(grad_out, out, x)` for `out = exp(x)`.

Requirements:
- Follow the canonical `(grad_out, out, x) -> grad_in` signature.
- Return `grad_out * out` — **use the cached `out`**, do NOT call `torch.exp`,
  `np.exp`, `math.exp`, or `**` with base e inside the function. The test
  passes a fake `out` and checks that the result tracks the fake (i.e. you
  trust `out`, not `x`).
- Return shape must equal `x.shape` (which equals `out.shape` for exp).


In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx4_exp_back(grad_out, out, x):
    """dL/dx for out = exp(x). MUST use cached out, not recompute exp(x)."""
    raise NotImplementedError()


def _test_cx4():
    # --- real-world case: out = exp(x), grad_out = ones → grad_in = out ---
    x = t.tensor([0.0, 1.0, 2.0])
    out = t.exp(x)
    g = cx4_exp_back(t.ones(3), out, x)
    assert g.shape == x.shape
    assert t.allclose(g, out), f'unit grad_out: {g} vs {out}'

    # --- non-unit grad_out: chain rule scales each entry ---
    grad_out = t.tensor([3.0, -2.0, 5.0])
    g = cx4_exp_back(grad_out, out, x)
    assert t.allclose(g, grad_out * out)

    # --- matrix shape ---
    rng = t.Generator().manual_seed(1)
    X = t.randn(3, 4, generator=rng)
    G = t.randn(3, 4, generator=rng)
    OUT = t.exp(X)
    g = cx4_exp_back(G, OUT, X)
    assert g.shape == (3, 4)
    assert t.allclose(g, G * OUT)

    # --- THE point: must use cached `out`, not recompute exp(x) ---
    # Pass a DELIBERATELY WRONG `out` (not equal to exp(x)) and verify the result
    # tracks the fake `out`. A recompute-from-x implementation would ignore it.
    fake_x = t.tensor([0.0, 0.0, 0.0])      # exp(0) is 1.0, but we lie:
    fake_out = t.tensor([0.25, 0.5, 0.75])
    got = cx4_exp_back(t.ones(3), fake_out, fake_x)
    assert t.allclose(got, fake_out), (
        f'implementation appears to recompute exp(x) rather than trust out: got {got}, '
        f'expected {fake_out} (chain rule on the fake out)'
    )

    # --- witness vs torch.autograd ---
    x_ref = t.tensor([-0.5, 0.0, 0.8, 1.5], requires_grad=True)
    t.exp(x_ref).sum().backward()
    out_cached = t.exp(x_ref.detach())
    ours = cx4_exp_back(t.ones(4), out_cached, x_ref.detach())
    assert t.allclose(ours, x_ref.grad, atol=1e-6)

    _dd_passed.add('cx4')

_test_cx4()

<details><summary>Show solution — cx4</summary>

```python
def cx4_exp_back(grad_out, out, x):
    # d/dx exp(x) = exp(x) = out (already cached by the forward pass).
    # Signature passes `out` precisely so we can reuse it — no recompute.
    return grad_out * out

```

Two atoms in one line: the parameter list IS the canonical back-fn signature
(`backward-fn-signature`), and `grad_out * out` is the cached-out form of
exp's chain rule (`exp-back`). The fake-out test fails any implementation
that quietly recomputes `t.exp(x)` instead of trusting `out`.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx4',
        'subtopics': ["Backprop: exp_back", "Backprop: backward fn signature"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()